# Material de Apoio 04 | Python Especialista — Internals, Performance, Tipagem e Concorrência

Notebook para estudo em nível especialista: dataclasses, protocolos, generics, async, threads, metaclasses, testes e padrões de projeto.

**Nível:** Especialista

## 1. Filosofia

Em nível especialista, o foco deixa de ser só `fazer funcionar` e passa a ser:

- legibilidade
- previsibilidade
- testabilidade
- performance quando necessário
- tipagem clara
- APIs bem desenhadas
- baixo acoplamento
- observabilidade


## 2. Data classes avançadas

`dataclass` reduz boilerplate e pode ficar mais eficiente com `slots=True`.

In [ ]:
from dataclasses import dataclass

@dataclass(slots=True, frozen=True)
class Product:
    id: int
    name: str
    price: float

product = Product(1, "Notebook", 3500.0)
print(product)
print(product.name)


## 3. `__slots__`

Restringe atributos dinâmicos e pode reduzir uso de memória.

In [ ]:
class RegularUser:
    def __init__(self, name, age):
        self.name = name
        self.age = age

class SlottedUser:
    __slots__ = ("name", "age")

    def __init__(self, name, age):
        self.name = name
        self.age = age

u1 = RegularUser("Leonardo", 26)
u2 = SlottedUser("Leonardo", 26)

print(u1.name, u2.name)


## 4. `property` para encapsulamento

Você controla leitura e escrita como se fosse atributo normal.

In [ ]:
class Temperature:
    def __init__(self, celsius: float):
        self._celsius = celsius

    @property
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, value: float) -> None:
        if value < -273.15:
            raise ValueError("Temperatura inválida.")
        self._celsius = value

    @property
    def fahrenheit(self) -> float:
        return self._celsius * 9 / 5 + 32

temp = Temperature(25)
print(temp.celsius)
print(temp.fahrenheit)

temp.celsius = 30
print(temp.fahrenheit)


## 5. Descritores

Base do funcionamento de `property`, métodos e diversos mecanismos internos.

In [ ]:
class PositiveNumber:
    def __set_name__(self, owner, name):
        self.private_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.private_name)

    def __set__(self, instance, value):
        if value < 0:
            raise ValueError("Valor deve ser positivo.")
        setattr(instance, self.private_name, value)

class Order:
    total = PositiveNumber()

    def __init__(self, total):
        self.total = total

order = Order(150)
print(order.total)


## 6. Protocolos com `typing.Protocol`

Permitem tipagem estrutural: importa o comportamento, não a herança.

In [ ]:
from typing import Protocol

class SupportsClose(Protocol):
    def close(self) -> None: ...

class FileResource:
    def close(self) -> None:
        print("Fechando recurso")

def shutdown(resource: SupportsClose) -> None:
    resource.close()

shutdown(FileResource())


## 7. Generics com `TypeVar`

APIs tipadas e reutilizáveis.

In [1]:
from typing import TypeVar, Iterable

T = TypeVar("T")

def first(items: Iterable[T]) -> T:
    for item in items:
        return item
    raise ValueError("Iterável vazio")

print(first([10, 20, 30]))
print(first(("a", "b", "c")))


10
a


## 8. `TypedDict`

Útil quando você quer tipar dicionários com estrutura conhecida.

In [ ]:
from typing import TypedDict

class UserPayload(TypedDict):
    id: int
    name: str
    email: str

payload: UserPayload = {
    "id": 1,
    "name": "Leonardo",
    "email": "leo@gmail.com"
}

print(payload["name"])


## 9. `Enum`

Evita strings soltas espalhadas no código.

In [ ]:
from enum import Enum

class Status(str, Enum):
    PENDING = "pending"
    APPROVED = "approved"
    REJECTED = "rejected"

status = Status.APPROVED
print(status)
print(status.value)


## 10. ABC — classes abstratas

Força contrato explícito.

In [2]:
from abc import ABC, abstractmethod

class PaymentProcessor(ABC):
    @abstractmethod
    def pay(self, amount: float) -> str:
        ...

class PixProcessor(PaymentProcessor):
    def pay(self, amount: float) -> str:
        return f"Pagamento PIX de R$ {amount:.2f} realizado"

processor = PixProcessor()
print(processor.pay(150.0))


Pagamento PIX de R$ 150.00 realizado


## 11. Context managers

Controle robusto de setup/teardown.

In [ ]:
class Transaction:
    def __enter__(self):
        print("Abrindo transação")
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc:
            print("Rollback")
        else:
            print("Commit")
        return False

with Transaction():
    print("Executando operação")


## 12. `contextlib.contextmanager`

Forma mais enxuta de criar context managers.

In [ ]:
from contextlib import contextmanager

@contextmanager
def managed_resource(name):
    print(f"abrindo {name}")
    try:
        yield name
    finally:
        print(f"fechando {name}")

with managed_resource("arquivo.txt") as resource:
    print("usando:", resource)


## 13. Iteradores e protocolo de iteração

Entender isso ajuda a dominar `for`, generators e streaming.

In [ ]:
class CountDown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current < 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value

for n in CountDown(3):
    print(n)


## 14. Generators para processamento lazy

Muito úteis para grandes volumes de dados.

In [ ]:
def read_fake_lines():
    for i in range(5):
        yield f"linha {i}"

lines = (line.upper() for line in read_fake_lines() if "2" not in line)

for line in lines:
    print(line)


## 15. `yield from`

Encaminha valores de outro iterável/generator.

In [3]:
def gen1():
    yield 1
    yield 2

def gen2():
    yield 0
    yield from gen1()
    yield 3

print(list(gen2()))


[0, 1, 2, 3]


## 16. Closures

Funções podem capturar estado externo.

In [ ]:
def make_multiplier(factor):
    def multiply(value):
        return value * factor
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(10))
print(triple(10))


## 17. Decorators com `functools.wraps`

Preservam metadados da função original.

In [ ]:
from functools import wraps

def logged(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@logged
def add(a, b):
    return a + b

print(add(2, 3))
print(add.__name__)


## 18. `singledispatch`

Despacho por tipo de forma organizada.

In [ ]:
from functools import singledispatch

@singledispatch
def serialize(value):
    return str(value)

@serialize.register
def _(value: int):
    return {"type": "int", "value": value}

@serialize.register
def _(value: list):
    return {"type": "list", "value": value}

print(serialize(10))
print(serialize([1, 2, 3]))
print(serialize("abc"))


## 19. Pattern matching (`match-case`)

Muito útil para parsing e fluxos com formato conhecido.

In [4]:
def handle_event(event):
    match event:
        case {"type": "user_created", "payload": {"id": user_id, "name": name}}:
            return f"novo usuário: {user_id} - {name}"
        case {"type": "order_paid", "payload": {"id": order_id, "total": total}}:
            return f"pedido pago: {order_id} - {total}"
        case _:
            return "evento desconhecido"

print(handle_event({"type": "user_created", "payload": {"id": 1, "name": "Leo"}}))
print(handle_event({"type": "order_paid", "payload": {"id": 99, "total": 350}}))


novo usuário: 1 - Leo
pedido pago: 99 - 350


## 20. `collections` na prática

Ferramentas subestimadas e muito úteis.

In [ ]:
from collections import Counter, defaultdict, deque

text = "banana"
print(Counter(text))

grouped = defaultdict(list)
for category, value in [("a", 1), ("a", 2), ("b", 3)]:
    grouped[category].append(value)
print(dict(grouped))

queue = deque([1, 2, 3])
queue.appendleft(0)
queue.append(4)
print(queue)


## 21. `heapq`

Estrutura eficiente para top-k e prioridade.

In [ ]:
import heapq

numbers = [10, 4, 7, 1, 99, 12, 3]
smallest_three = heapq.nsmallest(3, numbers)
largest_two = heapq.nlargest(2, numbers)

print(smallest_three)
print(largest_two)


## 22. `bisect`

Inserção e busca eficiente em listas ordenadas.

In [ ]:
import bisect

values = [10, 20, 30, 40]
bisect.insort(values, 25)

print(values)
print(bisect.bisect_left(values, 30))


## 23. `pathlib`

Melhor que manipular caminhos por string.

In [ ]:
from pathlib import Path

base = Path(".")
file_path = base / "dados" / "arquivo.csv"

print(file_path)
print(file_path.suffix)
print(file_path.name)


## 24. Logging estruturado

Especialista não depende só de `print()`.

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("app")

logger.info("aplicação iniciada")
logger.warning("uso alto de memória")


## 25. Exceções customizadas

Melhor semântica de domínio.

In [ ]:
class DomainError(Exception):
    pass

class InsufficientFundsError(DomainError):
    pass

def withdraw(balance: float, amount: float) -> float:
    if amount > balance:
        raise InsufficientFundsError("Saldo insuficiente.")
    return balance - amount

try:
    print(withdraw(100, 150))
except InsufficientFundsError as e:
    print(type(e).__name__, "-", e)


## 26. `ExceptionGroup` e `except*`

Útil em cenários concorrentes e validações múltiplas.

In [ ]:
def validate_user(data):
    errors = []

    if not data.get("name"):
        errors.append(ValueError("name obrigatório"))
    if data.get("age", 0) < 18:
        errors.append(PermissionError("idade mínima 18"))
    if "@" not in data.get("email", ""):
        errors.append(ValueError("email inválido"))

    if errors:
        raise ExceptionGroup("validation errors", errors)

try:
    validate_user({"name": "", "age": 16, "email": "invalido"})
except* ValueError as eg:
    print("ValueErrors:")
    for e in eg.exceptions:
        print("-", e)
except* PermissionError as eg:
    print("PermissionErrors:")
    for e in eg.exceptions:
        print("-", e)


## 27. Async básico com `asyncio`

I/O concorrente sem bloquear a thread principal.

In [ ]:
import asyncio

async def fetch_data(name, delay):
    await asyncio.sleep(delay)
    return f"{name} concluído"

async def main():
    results = await asyncio.gather(
        fetch_data("tarefa_1", 1),
        fetch_data("tarefa_2", 0.5),
        fetch_data("tarefa_3", 0.2),
    )
    print(results)

asyncio.run(main())


## 28. `asyncio.TaskGroup`

Estrutura moderna para grupos de tarefas.

In [ ]:
import asyncio

async def worker(name, delay):
    await asyncio.sleep(delay)
    print(f"{name} done")

async def main():
    async with asyncio.TaskGroup() as tg:
        tg.create_task(worker("a", 0.2))
        tg.create_task(worker("b", 0.4))
        tg.create_task(worker("c", 0.1))

asyncio.run(main())


## 29. Threads para I/O bound

Úteis quando a limitação é espera, não CPU.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def io_task(name):
    time.sleep(0.2)
    return f"{name} finalizada"

with ThreadPoolExecutor(max_workers=3) as executor:
    results = list(executor.map(io_task, ["t1", "t2", "t3"]))

print(results)


## 30. Processos para CPU bound

Use quando o gargalo for CPU pesada.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def square(x):
    return x * x

if __name__ == "__main__":
    with ProcessPoolExecutor() as executor:
        results = list(executor.map(square, [1, 2, 3, 4]))
    print(results)


## 31. Profiling simples com `timeit`

Meça antes de otimizar.

In [ ]:
import timeit

code_list_comp = "[x * 2 for x in range(1000)]"
code_loop = '''
result = []
for x in range(1000):
    result.append(x * 2)
'''

print("list comprehension:", timeit.timeit(code_list_comp, number=1000))
print("for loop:", timeit.timeit(code_loop, number=1000))


## 32. Caching com `lru_cache`

Muito útil para funções determinísticas.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(35))
print(fib.cache_info())


## 33. Modelagem imutável

Imutabilidade reduz bugs de estado.

In [ ]:
from dataclasses import dataclass, replace

@dataclass(frozen=True)
class Customer:
    id: int
    name: str
    active: bool = True

customer = Customer(1, "Leonardo")
updated = replace(customer, active=False)

print(customer)
print(updated)


## 34. Métodos especiais relevantes

`__repr__`, `__eq__`, `__hash__`, `__call__` e afins fazem diferença em APIs elegantes.

In [ ]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, value):
        return value * self.factor

    def __repr__(self):
        return f"Multiplier(factor={self.factor})"

double = Multiplier(2)
print(double)
print(double(10))


## 35. Metaclasses

Tema avançado: alteram a criação da classe. Use raro e com critério.

In [ ]:
class RegistryMeta(type):
    registry = {}

    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        if name != "BasePlugin":
            mcls.registry[name] = cls
        return cls

class BasePlugin(metaclass=RegistryMeta):
    pass

class PluginA(BasePlugin):
    pass

class PluginB(BasePlugin):
    pass

print(RegistryMeta.registry)


## 36. `__init_subclass__`

Muitas vezes é alternativa mais simples que metaclass.

In [ ]:
class PluginBase:
    registry = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        PluginBase.registry[cls.__name__] = cls

class CsvPlugin(PluginBase):
    pass

class JsonPlugin(PluginBase):
    pass

print(PluginBase.registry)


## 37. Validação com `__post_init__`

Boa prática em data classes.

In [ ]:
from dataclasses import dataclass

@dataclass
class Account:
    owner: str
    balance: float

    def __post_init__(self):
        if self.balance < 0:
            raise ValueError("Saldo inicial não pode ser negativo.")

account = Account("Leonardo", 100.0)
print(account)


## 38. Testes com `unittest`

Mesmo sem framework externo, Python já vem com base sólida.

In [ ]:
import unittest

def add(a, b):
    return a + b

class TestMath(unittest.TestCase):
    def test_add(self):
        self.assertEqual(add(2, 3), 5)

    def test_add_negative(self):
        self.assertEqual(add(-1, 1), 0)

suite = unittest.TestLoader().loadTestsFromTestCase(TestMath)
result = unittest.TextTestRunner(verbosity=2).run(suite)


## 39. `doctest`

Documentação executável para casos simples.

In [ ]:
import doctest

def multiply(a, b):
    '''
    >>> multiply(2, 3)
    6
    >>> multiply(-1, 4)
    -4
    '''
    return a * b

doctest.run_docstring_examples(multiply, globals(), verbose=True)


## 40. `operator`

Pequeno módulo útil e elegante.

In [ ]:
from operator import itemgetter, attrgetter

rows = [
    {"name": "b", "score": 10},
    {"name": "a", "score": 30},
    {"name": "c", "score": 20},
]

print(sorted(rows, key=itemgetter("score")))

class User:
    def __init__(self, name):
        self.name = name

users = [User("Carlos"), User("Ana"), User("Bruno")]
print([u.name for u in sorted(users, key=attrgetter("name"))])


## 41. Mini projeto — pipeline de processamento

Exemplo de composição com funções pequenas.

In [ ]:
from collections import Counter

def normalize(lines):
    for line in lines:
        yield line.strip().lower()

def tokenize(lines):
    for line in lines:
        for word in line.split():
            yield word

def remove_short(words, min_size=4):
    for word in words:
        if len(word) >= min_size:
            yield word

lines = [
    "Python é poderoso",
    "Python é flexível",
    "Dados com Python"
]

pipeline = remove_short(tokenize(normalize(lines)))
counts = Counter(pipeline)

print(counts)


## 42. Mini projeto — API de domínio bem tipada

In [ ]:
from dataclasses import dataclass
from enum import Enum

class OrderStatus(str, Enum):
    CREATED = "created"
    PAID = "paid"
    CANCELLED = "cancelled"

@dataclass(slots=True)
class Order:
    id: int
    total: float
    status: OrderStatus = OrderStatus.CREATED

    def pay(self) -> None:
        if self.status is not OrderStatus.CREATED:
            raise ValueError("Pedido não pode ser pago nesse estado.")
        self.status = OrderStatus.PAID

    def cancel(self) -> None:
        if self.status is OrderStatus.PAID:
            raise ValueError("Pedido pago não pode ser cancelado.")
        self.status = OrderStatus.CANCELLED

order = Order(1, 100.0)
order.pay()
print(order)


## 43. Checklist mental de especialista

Antes de escrever código, pense:

- isso precisa ser mutável?
- preciso de classe mesmo, ou função + dataclass resolve?
- a API está previsível?
- os nomes estão claros?
- há contrato explícito de tipos?
- o erro é tratado no nível certo?
- preciso otimizar? já medi?
- o fluxo é síncrono ou I/O bound?
- isso é fácil de testar?
- dá para quebrar em partes menores?


## 44. Exercícios de especialista

1. Implemente um cache TTL manual.  
2. Crie um decorator de retry com limite de tentativas.  
3. Modele um event bus simples com registro dinâmico de handlers.  
4. Faça um parser assíncrono de múltiplos arquivos.  
5. Crie uma classe `Money` com operadores (`+`, `-`, `==`).  
6. Implemente um `RateLimiter` como context manager.  
7. Crie um descriptor que valide string não vazia.  
8. Monte um pipeline lazy para ler, filtrar e agregar linhas.  
9. Escreva testes para uma classe de domínio.  
10. Reimplemente um registry usando `__init_subclass__`.  
